# Import necessary libraries

In [ ]:
import pandas as pd
import nltk
from textblob import TextBlob
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import ComplementNB, MultinomialNB, BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from collections import Counter
from wordcloud import WordCloud
import matplotlib.pyplot as plt
import re
import seaborn as sns

# Download required NLTK data

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Function to classify sentiment using TextBlob

In [ ]:
def classify_sentiment(text):
    if not isinstance(text, str):  # Ensure text is a string
        text = ""

    score = TextBlob(text).sentiment.polarity

    if score >= 0.05:
        return "Positive"
    elif score <= -0.05:
        return "Negative"
    else:
        return "Neutral"


# Load training and testing datasets

In [ ]:
# Load training and testing datasets
train_df = pd.read_csv('train_dataset.csv')
test_df = pd.read_csv('test_dataset.csv')

# Ensure all comments are strings before classification
train_df['Final_Comment'] = train_df['Final_Comment'].astype(str)
test_df['Final_Comment'] = test_df['Final_Comment'].astype(str)

# Apply sentiment classification to BOTH datasets
train_df['Sentiment'] = train_df['Final_Comment'].apply(classify_sentiment)
test_df['Sentiment'] = test_df['Final_Comment'].apply(classify_sentiment)

# Save updated datasets
train_df.to_csv('train_dataset_sentiment_textblob.csv', index=False)
test_df.to_csv('test_dataset_sentiment_textblob.csv', index=False)

print("✅ Sentiment classification using TextBlob completed for both datasets.")

# Convert text labels into numeric labels

In [ ]:
sentiment_mapping = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
train_df['Sentiment_Label'] = train_df['Sentiment'].map(sentiment_mapping)
test_df['Sentiment_Label'] = test_df['Sentiment'].map(sentiment_mapping)


# Verify if sentiment labels are successfully assigned

In [ ]:
print("Train dataset sentiment distribution:\n", train_df['Sentiment'].value_counts())
print("Test dataset sentiment distribution:\n", test_df['Sentiment'].value_counts())

# Feature Extraction using TF-IDF Vectorizer
vectorizer = TfidfVectorizer(max_features=5000)
X_train = vectorizer.fit_transform(train_df['Final_Comment'])
X_test = vectorizer.transform(test_df['Final_Comment'])

y_train = train_df['Sentiment_Label']
y_test = test_df['Sentiment_Label']


# Function to preprocess text

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return words

# Most common words


In [ ]:
train_words = train_df['Final_Comment'].apply(preprocess_text).sum()
train_common_words = Counter(train_words).most_common(10)
print("\n🔹 Most common words in training dataset:", train_common_words)

test_words = test_df['Final_Comment'].apply(preprocess_text).sum()
test_common_words = Counter(test_words).most_common(10)
print("\n🔹 Most common words in testing dataset:", test_common_words)

# Most common sentiment-specific words

In [ ]:
positive_words = train_df[train_df['Sentiment'] == 'Positive']['Final_Comment'].apply(preprocess_text).sum()
most_common_positive_word, _ = Counter(positive_words).most_common(1)[0]
print("\n✅ Most common word in Positive sentiment:", most_common_positive_word)

negative_words = train_df[train_df['Sentiment'] == 'Negative']['Final_Comment'].apply(preprocess_text).sum()
most_common_negative_word, _ = Counter(negative_words).most_common(1)[0]
print("\n✅ Most common word in Negative sentiment:", most_common_negative_word)


# Bag of Words


In [ ]:
vectorizer_bow = CountVectorizer(max_features=10, vocabulary=[most_common_positive_word, most_common_negative_word])
X_bow_train = vectorizer_bow.fit_transform(train_df['Final_Comment'])
bow_df = pd.DataFrame(X_bow_train.toarray(), columns=vectorizer_bow.get_feature_names_out())
print("\n📊 Bag of Words Representation (Top Words):\n", bow_df.head())


# Wordclouds

In [ ]:
def generate_wordcloud(words, title):
    wordcloud = WordCloud(width=800, height=400, background_color='white').generate(' '.join(words))
    plt.figure(figsize=(10, 5))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis("off")
    plt.title(title, fontsize=14)
    plt.show()

generate_wordcloud(positive_words, "Word Cloud for Positive Sentiment")
generate_wordcloud(negative_words, "Word Cloud for Negative Sentiment")


# Evaluation: Naive Bayes Models


# ComplementNB

In [ ]:
CNB = ComplementNB()
CNB.fit(X_train, y_train)
predicted_cnb = CNB.predict(X_test)
accuracy_cnb = accuracy_score(y_test, predicted_cnb)

print('✅ ComplementNB model accuracy:', '{:04.2f}'.format(accuracy_cnb * 100), '%')
print('------------------------------------------------')
print('Confusion Matrix:\n', pd.DataFrame(confusion_matrix(y_test, predicted_cnb)))
print('------------------------------------------------')
print('Classification Report:\n', classification_report(y_test, predicted_cnb))


# MultinomialNB

In [ ]:
MNB = MultinomialNB()
MNB.fit(X_train, y_train)
predicted_mnb = MNB.predict(X_test)
accuracy_mnb = accuracy_score(y_test, predicted_mnb)

print('✅ MultinomialNB model accuracy:', '{:04.2f}'.format(accuracy_mnb * 100), '%')
print('------------------------------------------------')
print('Confusion Matrix:\n', pd.DataFrame(confusion_matrix(y_test, predicted_mnb)))
print('------------------------------------------------')
print('Classification Report:\n', classification_report(y_test, predicted_mnb))


# BernoulliNB

In [ ]:
BNB = BernoulliNB()
BNB.fit(X_train, y_train)
predicted_bnb = BNB.predict(X_test)
accuracy_bnb = accuracy_score(y_test, predicted_bnb)

print('✅ BernoulliNB model accuracy:', '{:04.2f}'.format(accuracy_bnb * 100), '%')
print('------------------------------------------------')
print('Confusion Matrix:\n', pd.DataFrame(confusion_matrix(y_test, predicted_bnb)))
print('------------------------------------------------')
print('Classification Report:\n', classification_report(y_test, predicted_bnb))


# Plot Sentiment Distributions

In [ ]:
def plot_sentiment_distribution(df, title):
    plt.figure(figsize=(8,5))
    sns.countplot(x=df['Sentiment'], palette='coolwarm', order=['Positive', 'Neutral', 'Negative'])
    plt.title(title, fontsize=14)
    plt.xlabel("Sentiment", fontsize=12)
    plt.ylabel("Count", fontsize=12)
    plt.show()

plot_sentiment_distribution(train_df, "🔹 Sentiment Distribution in Training Dataset (TextBlob)")
plot_sentiment_distribution(test_df, "🔹 Sentiment Distribution in Testing Dataset (TextBlob)")


# Confusion Matrix Visuals


In [ ]:
def plot_confusion_matrix(y_test, y_pred, title):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Neutral', 'Positive'], yticklabels=['Negative', 'Neutral', 'Positive'])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.show()

plot_confusion_matrix(y_test, predicted_cnb, "🔹 ComplementNB Confusion Matrix")
plot_confusion_matrix(y_test, predicted_mnb, "🔹 MultinomialNB Confusion Matrix")
plot_confusion_matrix(y_test, predicted_bnb, "🔹 BernoulliNB Confusion Matrix")



# Top contributing words for ComplementNB


In [ ]:
feature_names = vectorizer.get_feature_names_out()
feature_importance = CNB.feature_log_prob_

for i, label in enumerate(["Negative", "Neutral", "Positive"]):
    top_features = sorted(zip(feature_importance[i], feature_names), reverse=True)[:10]
    words, scores = zip(*top_features)

    plt.figure(figsize=(8,4))
    sns.barplot(x=list(scores), y=list(words), palette='coolwarm')
    plt.title(f"🔹 Top Words Contributing to {label} Sentiment")
    plt.xlabel("Log Probability")
    plt.ylabel("Words")
    plt.show()


In [ ]:
# Word clouds for each sentiment category
for sentiment in ["Positive", "Neutral", "Negative"]:
    words = train_df[train_df['Sentiment'] == sentiment]['Final_Comment'].apply(preprocess_text).sum()
    generate_wordcloud(words, f"🔹 Word Cloud for {sentiment} Sentiment")